# N-CMAPSS DS05 v12 — everything in one run

**This supersedes v10 and v11. Run this one; ignore those two.** v11's search space already contained
every range v10 widened, so running them separately would spend three hours re-deriving the same
thing. What v10 had that v11 lacked — the branch ablation and the uncapped raw number — is folded in
here.

## Where we are

| | capped-55 | raw | skill |
|---|---|---|---|
| MSTSDN published | 3.950 | — | +0.785 |
| ours, v9 | 4.859 | 10.038 | +0.736 |
| ours, v8 | 5.015 | 10.131 | +0.727 |
| ours, v7 matched protocol | 5.137 | 10.268 | +0.721 |
| trivial floor | 18.393 | 23.142 | 0 |

Three searches across v8 and v9 — 255 evaluations, three algorithms — all plateaued at validation
3.6 when the fitted val-to-test line says 3.02 is needed to reach 3.950. Within the current
architecture family the room is gone.

But two things say the family itself was never properly explored. The v9 winner sits **at a wall in 8
of its 13 dimensions**, pressing in a consistent direction: fewer conv layers, wider first filter,
faster shrink, no dense head, higher learning rate, smaller batch. And the two largest effects ever
measured here — the time window at 4.7 RMSE and the virtual sensors at 3.6 — were **fixed constants
during every search**, found only because v7's factorial happened to test them. The best hyperparameter
any search ever found was worth 0.5.

So the room, if there is any, is in the constants and the walls. This notebook opens both.

## What runs, in order

**A. Branch ablation** at the incumbent — full hybrid against LSTM-only, CNN-only and no-dense-head,
paired seeds. It runs first because it answers the architectural question, needs nothing from the
search, and survives an interrupted run.

**B. Screening** of twelve previously fixed constants, one factor at a time, single seed.

**C. Confirmation** of the top movers at four seeds, paired against the incumbent.

**D. PSO** over the full 27-dimensional space, seeded with the incumbent and the confirmed combination.

**E. Stage 2** — paired t-test on fresh seeds decides what we report.

**F. Ablation re-check** at the final configuration, if it differs from the incumbent.

**G. The uncapped raw number** at 1-in-10, comparable to the 8.247 already in the thesis.

## The twelve constants being opened

| constant | was | now | why |
|---|---|---|---|
| T, window length | 30 | 10-60 | window-vs-none was worth 4.7; T itself never tuned |
| CNN readout | average pooling | avg / max / avg+max / attention | cross-cycle study found **max** helped by 0.496 |
| optimizer | Adam | Adam / AdamW / RMSprop / Nadam | MSTSDN uses **AdamW** |
| weight decay | none | 1e-8 to 1e-2, decoupled | L2 was the best hyperparameter on FD003; with Adam the two differ |
| bidirectional LSTM | off | on / off | our **battery** model uses it; DS05 never did |
| loss | MSE | MSE / Huber | heavy-tailed target |
| dilation | 1 | 1 / 2 / 4, exponential per layer | widens temporal receptive field cheaply |
| branch fusion | concatenate | concat / Conv1D over the concatenated sequence | the **Peng fusion** fixed FD003 |
| activation | ReLU | relu / elu / gelu / swish / tanh | inherited, never chosen |
| dropout | one shared rate | separate conv and dense | — |
| LR schedule | plateau | plateau / cosine / none | — |
| scaler | MinMax | minmax / standard / robust | — |

Topology stays parallel and VAL_UNITS stays as the benchmarks define it — both by decision.

## Budget and safety

Around 150 training runs, roughly five hours. Every part writes `ncmapss_v12_all_trials.csv` when it
finishes, and `RUN` at the top lets you skip or resume parts, so an interruption costs one part
rather than the night.

In [ ]:
import os, json, gc, time, random, math, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.metrics import mean_squared_error
from scipy.stats import ttest_rel, spearmanr
from scipy.stats.qmc import Sobol
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Flatten, Dot, Softmax,
                                     GlobalAveragePooling1D, GlobalMaxPooling1D, Rescaling)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler

RUN=dict(ablation=True, screening=True, confirm=True, pso=True,
         stage2=True, recheck=True, uncapped=True)
T_START=time.time()
def elapsed(): return '%.1f h'%((time.time()-T_START)/3600)

def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)

def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False

USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
HAS_ADAMW=True
try: _=tf.keras.optimizers.experimental.AdamW(learning_rate=1e-3,weight_decay=1e-4)
except Exception as e:
    HAS_ADAMW=False
    print('AdamW unavailable (%s) -> adamw falls back to Adam and is reported as such'%type(e).__name__)
print('TF',tf.__version__,'| device',DEVICE,'| AdamW',HAS_ADAMW)

## Data

Unscaled windows cached by T; the scaler is folded into the model as a fixed affine input layer.
All three scalers are affine maps, so one copy of the windows serves every scaler choice — no memory
multiplication and no risk of a stale rescaled array. Statistics come from training rows only, so
nothing about leakage changes. Larger T means fewer windows, so memory stays roughly flat across
the range.

In [ ]:
SEED=42; SUB=50; VAL_UNITS=[5,6]; CAP=55
EPOCHS=100; PATIENCE=15
T_LEVELS=[10,20,30,45,60]
N_PSO=48; SWARM=12
N_SEEDS_ABL=5; N_SEEDS_CONF=4; N_SEEDS2=5; SHORTLIST=2; N_TOP=5
set_seeds(SEED)

def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)

def load_flat(sub):
    with h5py.File('N-CMAPSS_DS05.h5','r') as h:
        Sd=read_sub(h['X_s_dev'],sub); Vd=read_sub(h['X_v_dev'],sub)
        Ad=read_sub(h['A_dev'],sub);   Yd=read_sub(h['Y_dev'],sub)
        St=read_sub(h['X_s_test'],sub);Vt=read_sub(h['X_v_test'],sub)
        At=read_sub(h['A_test'],sub);  Yt=read_sub(h['Y_test'],sub)
    out=dict(FD=np.hstack([Sd,Vd]),FT=np.hstack([St,Vt]),
             UD=Ad[:,0].astype(int),CD=Ad[:,1].astype(int),YD=Yd.ravel().astype(np.float32),
             UT=At[:,0].astype(int),CT=At[:,1].astype(int),YT=Yt.ravel().astype(np.float32))
    del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()
    return out

FL=load_flat(SUB)
FD,FT=FL['FD'],FL['FT']; UD,CD,YD=FL['UD'],FL['CD'],FL['YD']; UT,CT,YT=FL['UT'],FL['CT'],FL['YT']
TRMASK=~np.isin(UD,VAL_UNITS); VAMASK=np.isin(UD,VAL_UNITS); NFEAT=FD.shape[1]

TR=FD[TRMASK]
_mn,_mx=TR.min(0),TR.max(0); _rg=np.where((_mx-_mn)>1e-8,_mx-_mn,1.0)
_mu,_sd=TR.mean(0),TR.std(0); _sd=np.where(_sd>1e-8,_sd,1.0)
_q1,_md,_q3=np.percentile(TR,25,0),np.median(TR,0),np.percentile(TR,75,0)
_iqr=np.where((_q3-_q1)>1e-8,_q3-_q1,1.0)
SCALERS={'minmax':(1.0/_rg,-_mn/_rg),'standard':(1.0/_sd,-_mu/_sd),'robust':(1.0/_iqr,-_md/_iqr)}
SCALERS={k:(v[0].astype(np.float32),v[1].astype(np.float32)) for k,v in SCALERS.items()}
del TR; gc.collect()

def make_windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

_CACHE={}
def data_for(T):
    if T in _CACHE: return _CACHE[T]
    if len(_CACHE)>=3: _CACHE.pop(sorted(_CACHE)[0]); gc.collect()
    t0=time.time()
    Xtr,ytr=make_windows(FD,UD,CD,YD,TRMASK,T)
    Xva,yva=make_windows(FD,UD,CD,YD,VAMASK,T)
    Xte,yte=make_windows(FT,UT,CT,YT,np.ones(len(YT),bool),T)
    d=dict(Xtr=Xtr,ytr=np.minimum(ytr,CAP),Xva=Xva,yva=np.minimum(yva,CAP),Xte=Xte,
           yte_raw=yte,yte_cap=np.minimum(yte,CAP),n_train=len(Xtr),
           triv_cap=float(np.sqrt(mean_squared_error(np.minimum(yte,CAP),
                     np.full(len(yte),np.minimum(ytr,CAP).mean())))),
           triv_raw=float(np.sqrt(mean_squared_error(yte,np.full(len(yte),ytr.mean())))))
    _CACHE[T]=d
    print('   built T=%-2d  train %s | val %s | test %s  (%.0f s, %.0f MB)'
          %(T,format(len(Xtr),','),format(len(Xva),','),format(len(Xte),','),
            time.time()-t0,(Xtr.nbytes+Xva.nbytes+Xte.nbytes)/1e6))
    return d

D30=data_for(30); TRIV_CAP=D30['triv_cap']; TRIV_RAW=D30['triv_raw']; N_SMALL=D30['n_train']
print('trivial floors at T=30 -> capped %.3f | raw %.3f'%(TRIV_CAP,TRIV_RAW))
print('leakage check: train units %s | val units %s'%(sorted(set(UD[TRMASK])),VAL_UNITS))

In [ ]:
READOUTS=['avg','max','avgmax','attn']; OPTS=['adam','adamw','rmsprop','nadam']
ACTS=['relu','elu','gelu','swish','tanh']; SCHED=['plateau','cosine','none']; SCL=['minmax','standard','robust']

SPACE=[('T','cat',T_LEVELS),
       ('n_conv','int',0,4), ('f1','int',24,384), ('f_growth','float',0.25,2.0),
       ('kernel','cat',[1,2,3,5,7]), ('dilation','cat',[1,2,4]), ('use_pool','cat',[0,1]),
       ('readout','cat',READOUTS),
       ('n_lstm','int',1,2), ('lstm_units','int',24,320), ('bidir','cat',[0,1]),
       ('fusion','cat',['concat','conv']), ('fuse_filters','int',16,192),
       ('dropout_conv','float',0.0,0.5), ('dropout_dense','float',0.0,0.5),
       ('l2reg','log',1e-7,1e-1), ('weight_decay','log',1e-8,1e-2),
       ('dense1','int',0,256), ('dense2','int',0,256),
       ('activation','cat',ACTS),
       ('optimizer','cat',OPTS), ('lr','log',1e-4,2e-2), ('lr_sched','cat',SCHED),
       ('loss','cat',['mse','huber']), ('huber_delta','float',1.0,30.0),
       ('scaler','cat',SCL), ('batch','cat',[32,64,128,256,512,1024])]
D=len(SPACE); HPN=[s[0] for s in SPACE]

def decode(z):
    z=np.clip(np.asarray(z,float),0,1); out={}
    for zi,s in zip(z,SPACE):
        n,k=s[0],s[1]
        if k=='int':    out[n]=int(round(s[2]+zi*(s[3]-s[2])))
        elif k=='float':out[n]=float(s[2]+zi*(s[3]-s[2]))
        elif k=='log':  out[n]=float(np.exp(np.log(s[2])+zi*(np.log(s[3])-np.log(s[2]))))
        else:           out[n]=s[2][min(int(zi*len(s[2])),len(s[2])-1)]
    out['use_lstm']=1
    return out
def encode(kw):
    z=[]
    for s in SPACE:
        n,k=s[0],s[1]; v=kw[n]
        if k in ('int','float'): z.append((v-s[2])/(s[3]-s[2]))
        elif k=='log':           z.append((np.log(v)-np.log(s[2]))/(np.log(s[3])-np.log(s[2])))
        else:                    z.append((s[2].index(v)+0.5)/len(s[2]))
    return np.clip(np.array(z,float),0,1)
def nrmsd(a,b): return float(np.linalg.norm(np.asarray(a)-np.asarray(b))/np.sqrt(D))

# Incumbent: the v9 winner, with every newly opened constant at its OLD fixed value,
# so any change measured below is attributable to that constant and nothing else.
INC=dict(T=30, n_conv=2, f1=192, f_growth=0.5, kernel=5, dilation=1, use_pool=0, readout='avg',
         n_lstm=1, lstm_units=88, bidir=0, fusion='concat', fuse_filters=64,
         dropout_conv=0.3442517771420791, dropout_dense=0.3442517771420791,
         l2reg=3.831223699431197e-05, weight_decay=1e-8, dense1=8, dense2=0,
         activation='relu', optimizer='adam', lr=0.005000000000000002, lr_sched='plateau',
         loss='mse', huber_delta=10.0, scaler='minmax', batch=64, use_lstm=1)
_rt=decode(encode(INC))
bad=[k for k in HPN if _rt[k]!=INC[k] and
     (isinstance(INC[k],str) or abs(float(_rt[k])-float(INC[k]))>1e-4*max(1.0,abs(float(INC[k]))))]
print('incumbent round-trip:','OK' if not bad else 'MISMATCH '+str(bad))
print('search dimension:',D)

In [ ]:
def readout_of(x,mode,seed):
    if mode=='avg':    return GlobalAveragePooling1D()(x)
    if mode=='max':    return GlobalMaxPooling1D()(x)
    if mode=='avgmax': return Concatenate()([GlobalAveragePooling1D()(x),GlobalMaxPooling1D()(x)])
    s=Dense(1,activation='tanh',
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))(x)
    return Flatten()(Dot(axes=1)([Softmax(axis=1)(s),x]))

def make_opt(kw):
    lr=kw['lr']; o=kw['optimizer']
    if o=='adamw' and HAS_ADAMW:
        return tf.keras.optimizers.experimental.AdamW(learning_rate=lr,weight_decay=kw['weight_decay'])
    if o=='rmsprop': return tf.keras.optimizers.RMSprop(lr)
    if o=='nadam':   return tf.keras.optimizers.Nadam(lr)
    return tf.keras.optimizers.Adam(lr)

def build(kw,seed=SEED):
    T=kw['T']; act=kw['activation']
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    sc,of=SCALERS[kw['scaler']]
    inp=Input(shape=(T,NFEAT))
    x=Rescaling(scale=sc,offset=of)(inp)
    seq=(kw['fusion']=='conv'); branches=[]
    if kw['n_conv']>0:
        c=x
        for i in range(kw['n_conv']):
            f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
            dil=min(int(kw['dilation'])**i, max(1,int(c.shape[1])//2))
            c=Conv1D(f,kw['kernel'],padding='same',activation=act,dilation_rate=dil,**kk)(c)
            if kw['dropout_conv']>0: c=Dropout(kw['dropout_conv'],seed=seed)(c)
            # pooling would break length matching for sequence fusion, so it is off there
            if (not seq) and kw['use_pool'] and i<2 and int(c.shape[1])>2:
                c=MaxPooling1D(2,padding='same')(c)
        branches.append(c if seq else readout_of(c,kw['readout'],seed))
    if kw.get('use_lstm',1):
        r=x
        for j in range(kw['n_lstm']):
            cell=LSTM(kw['lstm_units'],return_sequences=(seq or j<kw['n_lstm']-1),**kk)
            r=Bidirectional(cell)(r) if kw['bidir'] else cell(r)
        branches.append(r)
    if not branches: raise ValueError('no branch enabled')
    if seq:
        h=branches[0] if len(branches)==1 else Concatenate(axis=-1)(branches)
        h=Conv1D(kw['fuse_filters'],3,padding='same',activation=act,**kk)(h)
        h=readout_of(h,kw['readout'],seed)
    else:
        h=branches[0] if len(branches)==1 else Concatenate()(branches)
    if kw['dense1']>=8:
        h=Dense(kw['dense1'],activation=act,**kk)(h)
        if kw['dropout_dense']>0: h=Dropout(kw['dropout_dense'],seed=seed)(h)
    if kw['dense2']>=16: h=Dense(kw['dense2'],activation=act,**kk)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    loss=tf.keras.losses.Huber(delta=kw['huber_delta']) if kw['loss']=='huber' else 'mse'
    m.compile(optimizer=make_opt(kw),loss=loss); return m

def cbs_for(kw,epochs,patience):
    cbs=[EarlyStopping('val_loss',patience=patience,restore_best_weights=True)]
    if kw['lr_sched']=='plateau':
        cbs.append(ReduceLROnPlateau('val_loss',factor=0.5,patience=max(2,patience//2),min_lr=1e-6))
    elif kw['lr_sched']=='cosine':
        lr0=kw['lr']
        cbs.append(LearningRateScheduler(lambda e,_l,lr0=lr0,E=epochs:
                                         float(1e-6+0.5*(lr0-1e-6)*(1+math.cos(math.pi*e/E)))))
    return cbs

TRIALS=[]
def evaluate(kw,seed=SEED,tag='',ds=None,epochs=EPOCHS,patience=PATIENCE):
    d=data_for(kw['T']) if ds is None else ds
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    try:
        with tf.device(DEVICE):
            m=build(kw,seed=seed)
            m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=epochs,
                  batch_size=kw['batch'],verbose=0,callbacks=cbs_for(kw,epochs,patience))
            pv=m.predict(d['Xva'],batch_size=8192,verbose=0).ravel()
            pt=m.predict(d['Xte'],batch_size=8192,verbose=0).ravel()
        val=float(np.sqrt(mean_squared_error(d['yva'],pv)))
        tc =float(np.sqrt(mean_squared_error(d['yte_cap'],np.clip(pt,0,CAP))))
        trw=float(np.sqrt(mean_squared_error(d['yte_raw'],pt)))
        npm=int(m.count_params())
    except Exception as e:
        print('   trial failed (%s: %s) -> penalised'%(type(e).__name__,str(e)[:60]))
        val,tc,trw,npm=1e3,np.nan,np.nan,0
    if not np.isfinite(val): val=1e3
    rec=dict(tag=tag,seed=seed,val=val,test_capped=tc,test_raw=trw,params=npm,
             secs=round(time.time()-t0,1)); rec.update(kw)
    TRIALS.append(rec); return val

def checkpoint(note=''):
    pd.DataFrame(TRIALS).to_csv('ncmapss_v12_all_trials.csv',index=False)
    print('[checkpoint] %-22s %d trials | elapsed %s'%(note,len(TRIALS),elapsed()))

print('incumbent, one seed:')
V0=evaluate(INC,SEED,'incumbent'); R0=TRIALS[-1]
print('  val %.3f | capped %.3f | raw %.3f | %d params | %.0f s'
      %(R0['val'],R0['test_capped'],R0['test_raw'],R0['params'],R0['secs']))
print('  v9 stage-2 reference: val 3.797 | capped 4.859 | raw 10.038')
print('  rough projection for the whole notebook: %.1f h'%(150*R0['secs']/3600))

## A — does the convolutional branch earn its place?

Four variants of the incumbent on the same five seeds, paired. `LSTM only` sets n_conv to 0;
`CNN only` disables the recurrent branch; `no dense head` removes both dense layers. This runs first
because it is the architectural question, it depends on nothing downstream, and it survives an
interrupted night.

If the full hybrid does not beat `LSTM only` at p<0.05, the convolutional branch is not earning its
name on DS05 and the thesis should say so — against FD003, where architecture was the largest effect
measured anywhere in this project at 7.07 RMSE. That contrast is a result about the two datasets.

In [ ]:
def run_ablation(base,label,seeds):
    variants=[('full hybrid',dict(base)),
              ('LSTM only',dict(base,n_conv=0)),
              ('CNN only',dict(base,use_lstm=0)),
              ('no dense head',dict(base,dense1=0,dense2=0))]
    store={}; rows=[]
    for nm,kw in variants:
        if nm=='CNN only' and kw['n_conv']==0:
            print('  %-16s skipped (no branch would remain)'%nm); continue
        if nm=='LSTM only' and base['n_conv']==0:
            print('  %-16s identical to the full model here'%nm); continue
        v,c,r=[],[],[]
        for s in seeds:
            evaluate(kw,s,'abl(%s):%s'%(label,nm)); t=TRIALS[-1]
            v.append(t['val']); c.append(t['test_capped']); r.append(t['test_raw'])
        store[nm]=dict(val=np.array(v),capped=np.array(c),raw=np.array(r))
        rows.append(dict(base=label,variant=nm,val=float(np.mean(v)),capped=float(np.mean(c)),
                         capped_sd=float(np.std(c,ddof=1)),raw=float(np.mean(r)),
                         params=int(t['params'])))
        print('  %-16s val %6.3f | capped %6.3f +/- %.3f | raw %7.3f | %8d params'
              %(nm,np.mean(v),np.mean(c),np.std(c,ddof=1),np.mean(r),t['params']))
    print()
    print('  paired against the full hybrid, same %d seeds:'%len(seeds))
    for nm in store:
        if nm=='full hybrid': continue
        d=store[nm]['capped']-store['full hybrid']['capped']
        tt,pp=ttest_rel(store[nm]['capped'],store['full hybrid']['capped'])
        print('    %-16s diff %+7.3f  SE %.3f  p %.4f  %s%s'
              %(nm,d.mean(),d.std(ddof=1)/math.sqrt(len(d)),pp,
                'worse than the hybrid' if d.mean()>0 else 'better than the hybrid',
                '  (SIGNIFICANT)' if pp<0.05 else '  (within noise)'))
    if 'LSTM only' in store:
        d=store['LSTM only']['capped']-store['full hybrid']['capped']
        _,pp=ttest_rel(store['LSTM only']['capped'],store['full hybrid']['capped'])
        print()
        if pp<0.05 and d.mean()>0:  print('  => the convolutional branch EARNS its place')
        elif pp<0.05 and d.mean()<0:print('  => the convolutional branch HURTS')
        else:                       print('  => the convolutional branch is NOT distinguishable from removing it')
    return pd.DataFrame(rows), store

if RUN['ablation']:
    SEEDS_A=[SEED+300+s for s in range(N_SEEDS_ABL)]
    AB_INC,_store=run_ablation(INC,'incumbent',SEEDS_A)
    AB_INC.to_csv('ncmapss_v12_ablation_incumbent.csv',index=False)
    checkpoint('ablation')
else:
    AB_INC=pd.DataFrame()

## B — screening the twelve constants, one at a time

Single seed, so this is a filter and not a result. Its job is to rank twelve factors cheaply enough
that Part C can afford to replicate the ones worth replicating.

In [ ]:
SCREEN=[]
for n in ['T','readout','optimizer','bidir','loss','dilation','fusion','activation','lr_sched','scaler']:
    s=[q for q in SPACE if q[0]==n][0]
    for lv in s[2]:
        if lv==INC[n]: continue
        kw=dict(INC); kw[n]=lv
        if n=='optimizer' and lv=='adamw': kw['weight_decay']=1e-4
        SCREEN.append(('%s=%s'%(n,lv),n,lv,kw))
for wd in [1e-6,1e-4,1e-2]:
    kw=dict(INC); kw['optimizer']='adamw'; kw['weight_decay']=wd
    SCREEN.append(('weight_decay=%.0e'%wd,'weight_decay',wd,kw))
for dd in [0.0,0.15]:
    kw=dict(INC); kw['dropout_dense']=dd
    SCREEN.append(('dropout_dense=%.2f'%dd,'dropout_dense',dd,kw))

if RUN['screening']:
    print('screening %d variants against the incumbent (val %.3f, capped %.3f)'
          %(len(SCREEN),V0,R0['test_capped']))
    rows=[]
    for label,fac,lv,kw in SCREEN:
        y=evaluate(kw,SEED,'screen:'+label); t=TRIALS[-1]
        rows.append(dict(label=label,factor=fac,level=str(lv),val=y,d_val=y-V0,
                         capped=t['test_capped'],d_capped=t['test_capped']-R0['test_capped'],
                         raw=t['test_raw'],params=t['params'],secs=t['secs']))
        print('  %-24s val %7.3f (%+6.3f)   capped %6.3f (%+6.3f)  %4.0fs'
              %(label,y,y-V0,t['test_capped'],t['test_capped']-R0['test_capped'],t['secs']))
    SC=pd.DataFrame(rows).sort_values('d_val')
    SC.to_csv('ncmapss_v12_screening.csv',index=False)
    print()
    print('best single changes on validation:')
    print(SC.head(8)[['label','val','d_val','capped','d_capped']].to_string(index=False))
    print()
    print('effect size by factor (best level minus incumbent, validation):')
    for f,g in SC.groupby('factor'):
        b=g.nsmallest(1,'d_val').iloc[0]
        print('   %-14s %+7.3f   best level %s'%(f,b['d_val'],b['level']))
    checkpoint('screening')
else:
    SC=pd.read_csv('ncmapss_v12_screening.csv')

## C — confirm the movers on fresh seeds

Screening ranked on one seed, which is exactly the setting where argmin is biased low. The five best
single-factor changes are re-run at four seeds and paired against the incumbent at the same seeds.
Only what survives goes into the combined configuration.

In [ ]:
def apply_factor(base,f,lv):
    kw=dict(base)
    if f in ('weight_decay','dropout_dense'):
        kw[f]=float(lv)
        if f=='weight_decay': kw['optimizer']='adamw'
    elif f in ('T','dilation','bidir'): kw[f]=int(lv)
    else: kw[f]=lv
    return kw

if RUN['confirm']:
    TOP=SC.nsmallest(N_TOP,'d_val')
    SEEDS_C=[SEED+600+s for s in range(N_SEEDS_CONF)]
    cands=[('incumbent',dict(INC))]
    for _,row in TOP.iterrows():
        cands.append((row['label'],apply_factor(INC,row['factor'],row['level'])))
    CONF={}; rows=[]
    for name,kw in cands:
        v,c,r=[],[],[]
        for s in SEEDS_C:
            evaluate(kw,s,'confirm:'+name); t=TRIALS[-1]
            v.append(t['val']); c.append(t['test_capped']); r.append(t['test_raw'])
        CONF[name]=dict(val=np.array(v),capped=np.array(c),raw=np.array(r))
        rows.append(dict(name=name,val=float(np.mean(v)),val_sd=float(np.std(v,ddof=1)),
                         capped=float(np.mean(c)),capped_sd=float(np.std(c,ddof=1)),
                         raw=float(np.mean(r))))
        print('  %-24s val %6.3f+/-%.3f | capped %6.3f+/-%.3f'
              %(name,np.mean(v),np.std(v,ddof=1),np.mean(c),np.std(c,ddof=1)))
    CF=pd.DataFrame(rows); CF.to_csv('ncmapss_v12_confirm.csv',index=False)
    print()
    print('paired against the incumbent, same %d seeds:'%N_SEEDS_CONF)
    KEEP=[]
    for name in CONF:
        if name=='incumbent': continue
        d=CONF[name]['capped']-CONF['incumbent']['capped']
        tt,pp=ttest_rel(CONF[name]['capped'],CONF['incumbent']['capped'])
        sig=bool(pp<0.05 and d.mean()<0)
        if sig: KEEP.append(name)
        print('  %-24s diff %+7.3f  SE %.3f  p %.4f %s'
              %(name,d.mean(),d.std(ddof=1)/math.sqrt(len(d)),pp,'  CONFIRMED' if sig else ''))
    print()
    print('confirmed:',KEEP if KEEP else 'none survived replication')
    COMBO=dict(INC)
    for name in KEEP:
        row=TOP[TOP.label==name].iloc[0]
        COMBO=apply_factor(COMBO,row['factor'],row['level'])
    diffs=[k for k in HPN if COMBO[k]!=INC[k]]
    print('combined configuration differs from the incumbent in:',diffs or 'nothing')
    checkpoint('confirm')
else:
    COMBO=dict(INC); KEEP=[]

## D — PSO over all 27 dimensions

Seeded with the incumbent and the confirmed combination, so the swarm starts from the best two points
we know and cannot return worse. BO is not run: it lost to random search at equal budget in v8 and
again in v9, and a third replication buys nothing the paper does not already have.

27 dimensions at 48 evaluations is underpowered, and this part should be read as what a modest budget
finds rather than as a map of the space. That is exactly why Parts B and C come first.

In [ ]:
def far_design(n,refs,min_d=0.35,seed=SEED):
    eng=Sobol(d=D,scramble=True,seed=seed); pts=[]; guard=0
    while len(pts)<n and guard<50:
        guard+=1
        for c in eng.random(max(8,n*4)):
            if min(nrmsd(c,q) for q in refs)>=min_d:
                pts.append(c)
                if len(pts)==n: break
    rs=np.random.RandomState(seed)
    while len(pts)<n:
        c=rs.rand(D)
        if min(nrmsd(c,q) for q in refs)>=min_d: pts.append(c)
    return np.array(pts[:n])

if RUN['pso']:
    Z_INC=encode(INC); Z_COMBO=encode(COMBO)
    ITERS=N_PSO//SWARM; rng=np.random.RandomState(SEED+7)
    P=far_design(SWARM,[Z_INC,Z_COMBO],seed=SEED+7)
    P[0]=Z_INC.copy(); P[1]=Z_COMBO.copy()
    V=(rng.rand(SWARM,D)-0.5)*0.30
    pbest=P.copy(); pbest_y=np.full(SWARM,np.inf); gbest=None; gbest_y=np.inf
    t0=time.time()
    for it in range(ITERS):
        w_in=0.85-0.5*it/max(1,ITERS-1)
        for k in range(SWARM):
            y=evaluate(decode(P[k]),SEED,'pso')
            if y<pbest_y[k]: pbest_y[k]=y; pbest[k]=P[k].copy()
            if y<gbest_y:    gbest_y=y;    gbest=P[k].copy()
        r1,r2=rng.rand(SWARM,D),rng.rand(SWARM,D)
        V=np.clip(w_in*V+1.5*r1*(pbest-P)+1.5*r2*(gbest-P),-0.25,0.25); P=np.clip(P+V,0,1)
        print('  pso iter %d/%d  gbest %7.3f  spread %.3f  elapsed %s'
              %(it+1,ITERS,gbest_y,float(np.mean(np.std(P,axis=0))),elapsed()))
    print('PSO done in %.1f min | best val %.3f'%((time.time()-t0)/60,gbest_y))
    GB=decode(gbest)
    print('gbest differs from the incumbent in:')
    for n in HPN:
        if GB[n]!=INC[n]: print('   %-14s %s  ->  %s'%(n,INC[n],GB[n]))
    checkpoint('pso')
else:
    GB=dict(COMBO)

## E — stage 2, and the standing table

In [ ]:
def kw_of(row):
    g=(lambda n: getattr(row,n)) if hasattr(row,'_fields') else (lambda n: row[n])
    out={}
    for s in SPACE:
        n,k=s[0],s[1]; v=g(n)
        out[n]=int(v) if k=='int' else (float(v) if k in ('float','log') else v)
    out['use_lstm']=1
    return out

if RUN['stage2']:
    SR=pd.DataFrame(TRIALS); SR=SR[(SR.tag=='pso')&(SR.val<999)]
    short=SR.nsmallest(SHORTLIST*3,'val').drop_duplicates(subset=HPN).head(SHORTLIST) if len(SR) else SR
    SEEDS2=[SEED+700+s for s in range(N_SEEDS2)]
    cands=[('incumbent',dict(INC)),('combo',dict(COMBO))]
    for i,q in enumerate(short.itertuples()): cands.append(('pso_%d'%(i+1),kw_of(q)))
    PER={}; STAGE2=[]; CAND={}
    for name,kw in cands:
        CAND[name]=dict(kw); v,c,r=[],[],[]
        for s in SEEDS2:
            evaluate(kw,s,'stage2:'+name); t=TRIALS[-1]
            v.append(t['val']); c.append(t['test_capped']); r.append(t['test_raw'])
        PER[name]=dict(val=np.array(v),capped=np.array(c),raw=np.array(r))
        STAGE2.append(dict(name=name,val=float(np.mean(v)),val_sd=float(np.std(v,ddof=1)),
                           capped=float(np.mean(c)),capped_sd=float(np.std(c,ddof=1)),
                           raw=float(np.mean(r)),raw_sd=float(np.std(r,ddof=1))))
        print('  %-12s val %6.3f+/-%.3f | capped %6.3f+/-%.3f | raw %7.3f+/-%.3f'
              %(name,np.mean(v),np.std(v,ddof=1),np.mean(c),np.std(c,ddof=1),np.mean(r),np.std(r,ddof=1)))
    S2=pd.DataFrame(STAGE2); S2.to_csv('ncmapss_v12_stage2.csv',index=False)
    win=S2[S2['name']!='incumbent'].nsmallest(1,'val').iloc[0]['name']
    d=PER[win]['capped']-PER['incumbent']['capped']
    tt,pp=ttest_rel(PER[win]['capped'],PER['incumbent']['capped'])
    print()
    print('validation selects: %s   (selection never saw a test score)'%win)
    print('paired vs incumbent: diff %+7.3f  SE %.3f  t %+6.2f  p %.4f'
          %(d.mean(),d.std(ddof=1)/math.sqrt(len(d)),tt,pp))
    ADOPT=bool(d.mean()<0 and pp<0.05)
    print('verdict:','ADOPT' if ADOPT else 'KEEP the incumbent (difference within noise)')
    FINAL_NAME=win if ADOPT else 'incumbent'; FINAL_KW=dict(CAND[FINAL_NAME])
    fin=S2[S2['name']==FINAL_NAME].iloc[0]
    print()
    print('%-42s %9s %9s %8s'%('','capped-55','raw','skill'))
    for lab,cc,rr in [('MSTSDN published',3.950,None),('ours, v12',fin.capped,fin.raw),
                      ('ours, v9',4.859,10.038),('ours, v8',5.015,10.131),
                      ('ours, v7 matched protocol',5.137,10.268),('trivial floor',TRIV_CAP,TRIV_RAW)]:
        print('%-42s %9.3f %9s %+8.3f'%(lab,cc,('%.3f'%rr) if rr is not None else '  --  ',1-cc/TRIV_CAP))
    print('remaining gap to MSTSDN: %+.3f'%(fin.capped-3.95))
    ok=pd.DataFrame(TRIALS); ok=ok[ok.val<999].dropna(subset=['test_capped'])
    RHO,_=spearmanr(ok.val,ok.test_capped)
    print('Spearman(val, test) over %d trials in 27 dimensions: %+.3f   [v9: +0.849]'%(len(ok),RHO))
    checkpoint('stage2')
else:
    FINAL_NAME='incumbent'; FINAL_KW=dict(INC); ADOPT=False; RHO=float('nan')

## F — ablation re-check at the final configuration

Part A answered the question at the incumbent. If the search moved somewhere materially different,
the answer could differ there too, so it is re-run rather than assumed.

In [ ]:
if RUN['recheck'] and FINAL_NAME!='incumbent':
    print('final configuration differs from the incumbent -> re-running the ablation there')
    AB_FIN,_=run_ablation(FINAL_KW,'final',[SEED+800+s for s in range(N_SEEDS_ABL)])
    AB_FIN.to_csv('ncmapss_v12_ablation_final.csv',index=False)
    checkpoint('recheck')
else:
    AB_FIN=pd.DataFrame()
    print('final configuration is the incumbent -> Part A already answers this, nothing to re-run')

## G — the uncapped raw number at 1-in-10

The thesis quotes 8.247 raw for DS05, measured on the old 18-feature snapshot at 1-in-10 sampling.
v7 put the 28-sensor windowed representation at 8.112 raw using a fifth of that data. This runs the
final configuration on that representation at 1-in-10 with **capping switched off**, which is the
figure comparable to what is already written.

**Budget is matched in gradient updates, not epochs.** Five times the data means five times the
updates per epoch, so the epoch count is divided by the data ratio. Holding epochs fixed would
quietly give this run five times the training of everything it is compared against — the mirror
image of the 113x under-training found in the original N-CMAPSS rebuild.

If the windows will not fit in memory the cell falls back to 1-in-20 and says which sampling it used.
A declared fallback is honest; a MemoryError five hours in is not.

In [ ]:
if RUN['uncapped']:
    Tf=int(FINAL_KW['T'])
    _CACHE.clear(); gc.collect()
    def big_dataset(sub,T):
        L=load_flat(sub)
        trm=~np.isin(L['UD'],VAL_UNITS); vam=np.isin(L['UD'],VAL_UNITS)
        Xtr,ytr=make_windows(L['FD'],L['UD'],L['CD'],L['YD'],trm,T)
        Xva,yva=make_windows(L['FD'],L['UD'],L['CD'],L['YD'],vam,T)
        Xte,yte=make_windows(L['FT'],L['UT'],L['CT'],L['YT'],np.ones(len(L['YT']),bool),T)
        del L; gc.collect()
        return dict(Xtr=Xtr,ytr=ytr,Xva=Xva,yva=yva,Xte=Xte,yte_raw=yte,
                    yte_cap=np.minimum(yte,CAP),n_train=len(Xtr),
                    triv_raw=float(np.sqrt(mean_squared_error(yte,np.full(len(yte),ytr.mean())))))
    SUB_BIG=10
    try:
        DB=big_dataset(SUB_BIG,Tf)
    except MemoryError:
        print('MemoryError at 1-in-%d -> falling back to 1-in-20'%SUB_BIG)
        SUB_BIG=20; gc.collect(); DB=big_dataset(SUB_BIG,Tf)
    ratio=DB['n_train']/float(N_SMALL)
    EP=max(10,int(round(EPOCHS/ratio))); PA=max(4,int(round(PATIENCE/ratio)))
    print('1-in-%d uncapped, T=%d | train %s windows (%.1fx) | test %s | trivial raw %.3f'
          %(SUB_BIG,Tf,format(DB['n_train'],','),ratio,format(len(DB['yte_raw']),','),DB['triv_raw']))
    print('update-matched budget: %d epochs, patience %d (was %d / %d)'%(EP,PA,EPOCHS,PATIENCE))
    raws,caps=[],[]
    for s in range(5):
        evaluate(FINAL_KW,SEED+900+s,'uncapped_1in%d'%SUB_BIG,ds=DB,epochs=EP,patience=PA)
        t=TRIALS[-1]; raws.append(t['test_raw']); caps.append(t['test_capped'])
        print('  seed %d  raw %7.3f | capped %6.3f | %4.0f s'%(SEED+900+s,t['test_raw'],t['test_capped'],t['secs']))
    raws=np.array(raws); caps=np.array(caps); fr=DB['triv_raw']
    print()
    print('%-52s %9s %8s'%('','raw RMSE','skill'))
    for lab,v in [('thesis figure, 18-feature snapshot at 1-in-10',8.247),
                  ('v7, 28 sensors windowed, uncapped, 1-in-50',8.112),
                  ('v12, final config, uncapped, 1-in-%d'%SUB_BIG,float(raws.mean())),
                  ('trivial floor',fr)]:
        print('%-52s %9.3f %+8.3f'%(lab,v,1-v/fr))
    print('v12 uncapped: %.3f +/- %.3f over 5 seeds'%(raws.mean(),raws.std(ddof=1)))
    print('(capped metric on the same runs, for reference: %.3f +/- %.3f)'%(caps.mean(),caps.std(ddof=1)))
    UNC=dict(sub=SUB_BIG,T=Tf,epochs=EP,raw_mean=float(raws.mean()),raw_sd=float(raws.std(ddof=1)),
             capped_mean=float(caps.mean()),trivial_raw=float(fr))
else:
    UNC={}

json.dump(dict(final_name=FINAL_NAME,adopt=ADOPT,
               final_kw={k:str(v) for k,v in FINAL_KW.items()},
               confirmed=KEEP,spearman=RHO,
               ablation_incumbent=AB_INC.to_dict('records') if len(AB_INC) else [],
               ablation_final=AB_FIN.to_dict('records') if len(AB_FIN) else [],
               uncapped=UNC,trivial_capped=TRIV_CAP,trivial_raw=TRIV_RAW),
          open('ncmapss_v12_summary.json','w'),indent=2,default=str)
checkpoint('done')
print('total elapsed:',elapsed())
print('saved ncmapss_v12_* files in',os.getcwd())